# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata safely as attributes
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

> **Note:** All dataset entities are referenced by their Croissant `@id` for clarity and reproducibility.

In [ ]:
# List all available record sets and their fields
record_sets = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | name: {field.name} | dataType: {getattr(field, 'data_type', 'unknown')}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We select all record sets and load the data using their Croissant `@id` identifiers.

In [ ]:
# Extract data from each record set and store DataFrames in a dictionary keyed by record set @id
record_set_ids = [rs.id for rs in record_sets]
dfs = {}

for rs_id in record_set_ids:
    print(f"Loading records from RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dfs[rs_id] = df
        print(f"  Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
    else:
        print("  No records loaded.")

if dfs:
    # List columns of the primary record set (use the first as example)
    primary_rs_id = record_set_ids[0]
    print(f"\nColumns for RecordSet {primary_rs_id}:")
    print(dfs[primary_rs_id].columns.tolist())
    display(dfs[primary_rs_id].head())
else:
    print("No data could be loaded from the record sets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records by specific numeric criteria, normalizing fields, and grouping/categorizing data. For each operation, fields and groupings are specified by Croissant field `@id`.

We'll demonstrate with the first record set (if available) and use its numeric fields for filtering and normalization.

In [ ]:
if dfs:
    df = dfs[primary_rs_id].copy()

    # Find a numeric field for demonstration
    rs = record_sets[0]
    numeric_field_id = None
    group_field_id = None
    for field in rs.fields:
        if getattr(field, 'data_type', '').lower() in ('float', 'number', 'integer'):
            if field.id in df.columns:
                numeric_field_id = field.id
                break
    # Just use the second field as a group-by key if possible
    for field in rs.fields:
        if field.id in df.columns and getattr(field, 'data_type', '').lower() in ('text', 'string') and field.id != numeric_field_id:
            group_field_id = field.id
            break
    if numeric_field_id is not None:
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        print(f"Field unique values: {df[numeric_field_id].unique()[:10]}")

        # Filter records with values > threshold (set as median for demonstration)
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            threshold = df[numeric_field_id].median()
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold} (median value): {len(filtered_df)} rows")
            display(filtered_df.head())

            # Normalize
            normalized_col = f"{numeric_field_id}_normalized"
            filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, normalized_col]].head())

            # Group by a string/text/cat field if available
            if group_field_id is not None:
                grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
                display(grouped.head())
        else:
            print(f"{numeric_field_id} is not a numeric column in the dataframe.")
    else:
        print("No numeric field found in the first record set for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will demonstrate a histogram of the selected numeric field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, color='cornflowerblue')
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet {primary_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10, 5))
        sns.boxplot(y=df[numeric_field_id], x=df[group_field_id])
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load, overview, and analyze a dataset using the `mlcroissant` library:
- Loaded dataset metadata and described its structure.
- Listed all record sets and their available fields, referencing all by their Croissant `@id`.
- Extracted data and performed exploratory data analysis, including filtering and normalization operations by field `@id`.
- Visualized data distributions with histograms and boxplots, again using field and record set `@id`s.

The `mlcroissant` library, alongside the FAIR (Findable, Accessible, Interoperable, Reusable) Croissant schema, enables programmatic analysis, reproducibility, and clear referencing of dataset structure for downstream scientific work.

For more advanced analysis (e.g., multi-table joins, text feature extraction, or machine learning), refer to the full schema documentation and the mlcroissant documentation: https://mlcommons.github.io/croissant/